In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: c:\QuantBricker
Fixed Income Library is loaded.


### Build yield curve sub model

In [2]:
path = "serialized/yc_model_calibrated.pickle"
yc_model = qfReadModelFromFile(path)
yc_model.components_.keys()

dict_keys(['SOFR-1B-FLAT', 'SOFRON Actual/360'])

### Create Build Method Collection

In [3]:
build_method_type = 'IR_SABR'
content = {
    'TARGET' : 'SOFR-1B-SWAPTION',
    'SWAPTION NORMAL VOLATILITY' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR BETA' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR NU' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR RHO' : 'USD-SOFR-SWAPTION',
    'VOL INTERPOLATION DOMAIN' : 'NORMAL VOLATILITY',
    'INTERPOLATION METHOD' : 'LINEAR',
    'EXTRAPOLATION METHOD' : 'FLAT',
    'BUSINESSDAY CONVENTION' : 'F',
    'HOLIDAY CONVENTION' : 'USGS',
    'SHIFT' : 0.04
}
sabr_build_method = qfCreateBuildMethod(build_method_type, content)
display(sabr_build_method.display())

,Name,Value
0,TARGET,SOFR-1B-SWAPTION
1,SWAPTION NORMAL VOLATILITY,USD-SOFR-SWAPTION
2,SWAPTION SABR BETA,USD-SOFR-SWAPTION
3,SWAPTION SABR NU,USD-SOFR-SWAPTION
4,SWAPTION SABR RHO,USD-SOFR-SWAPTION
5,VOL INTERPOLATION DOMAIN,NORMAL VOLATILITY
6,INTERPOLATION METHOD,LINEAR
7,EXTRAPOLATION METHOD,FLAT
8,BUSINESSDAY CONVENTION,F
9,HOLIDAY CONVENTION,USGS


In [4]:
### serialize / de-serialize
path = 'sabr_build_method.pickle'
qfWriteBuildMethodToFile(sabr_build_method, path)
display(sabr_build_method.display())
sabr_build_method_back = qfReadBuildMethodFromFile(path)
# check
display(sabr_build_method_back.display())
# house keeping
os.remove(path)

,Name,Value
0,TARGET,SOFR-1B-SWAPTION
1,SWAPTION NORMAL VOLATILITY,USD-SOFR-SWAPTION
2,SWAPTION SABR BETA,USD-SOFR-SWAPTION
3,SWAPTION SABR NU,USD-SOFR-SWAPTION
4,SWAPTION SABR RHO,USD-SOFR-SWAPTION
5,VOL INTERPOLATION DOMAIN,NORMAL VOLATILITY
6,INTERPOLATION METHOD,LINEAR
7,EXTRAPOLATION METHOD,FLAT
8,BUSINESSDAY CONVENTION,F
9,HOLIDAY CONVENTION,USGS


,Name,Value
0,EXTRAPOLATION METHOD,FLAT
1,INTERPOLATION METHOD,LINEAR
2,BUSINESSDAY CONVENTION,F
3,HOLIDAY CONVENTION,USGS
4,VOL INTERPOLATION DOMAIN,NORMAL VOLATILITY
5,SWAPTION SABR RHO,USD-SOFR-SWAPTION
6,SWAPTION SABR BETA,USD-SOFR-SWAPTION
7,SWAPTION SABR NU,USD-SOFR-SWAPTION
8,SHIFT,0.04
9,SWAPTION NORMAL VOLATILITY,USD-SOFR-SWAPTION


In [5]:
### caplet
build_method_type = 'IR_SABR'
content_cf = {
    'TARGET' : 'SOFR-1B-CAPFLOOR',
    'SWAPTION NORMAL VOLATILITY' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR BETA' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR NU' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR RHO' : 'USD-SOFR-CAPFLOOR',
    'VOL INTERPOLATION DOMAIN' : 'NORMAL VOLATILITY',
    'INTERPOLATION METHOD' : 'LINEAR',
    'EXTRAPOLATION METHOD' : 'FLAT',
    'BUSINESSDAY CONVENTION' : 'F',
    'HOLIDAY CONVENTION' : 'USGS',
    'SHIFT' : 0.04
}
sabr_build_method_cf = qfCreateBuildMethod(build_method_type, content_cf)
display(sabr_build_method_cf.display())

,Name,Value
0,TARGET,SOFR-1B-CAPFLOOR
1,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR
2,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR
3,SWAPTION SABR NU,USD-SOFR-CAPFLOOR
4,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR
5,VOL INTERPOLATION DOMAIN,NORMAL VOLATILITY
6,INTERPOLATION METHOD,LINEAR
7,EXTRAPOLATION METHOD,FLAT
8,BUSINESSDAY CONVENTION,F
9,HOLIDAY CONVENTION,USGS


In [6]:
### pack up as build method collection
bm_list = [sabr_build_method, sabr_build_method_cf]
bm_collection = qfCreateModelBuildMethodCollection(bm_list)
bm_collection.display()

,Name,Value
0,IR_SABR,SOFR-1B-SWAPTION
1,IR_SABR,SOFR-1B-CAPFLOOR


### Data Collection

In [7]:
### swaption
expiries = ['3M', '6M', '1Y', '5Y', '10Y']
tenors = ['1Y', '5Y', '10Y']
data_conv = 'USD-SOFR-SWAPTION'
# nv
df = pd.DataFrame(np.random.uniform(low=90./1e4, high=110./1e4, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nv = qfCreateData2D('Swaption Normal Volatility', data_conv, df)
# beta
df = pd.DataFrame(np.random.uniform(low=60./1e2, high=60./1e2, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_beta = qfCreateData2D('Swaption SABR Beta', data_conv, df)
# nu
df = pd.DataFrame(np.random.uniform(low=0.01, high=0.3, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nu= qfCreateData2D('Swaption SABR Nu', data_conv, df)
# rho
df = pd.DataFrame(np.random.uniform(low=-0.99, high=0.99, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_rho = qfCreateData2D('Swaption SABR Rho', data_conv, df)

In [8]:
### capfloor
expiries = ['3M', '6M', '1Y', '5Y', '10Y']
tenors = ['1M', '3M', '6M']
data_conv = 'USD-SOFR-CAPFLOOR'
# nv
df = pd.DataFrame(np.random.uniform(low=90./1e4, high=110./1e4, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nv_cf = qfCreateData2D('Swaption Normal Volatility', data_conv, df)
# beta
df = pd.DataFrame(np.random.uniform(low=60./1e2, high=60./1e2, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_beta_cf = qfCreateData2D('Swaption SABR Beta', data_conv, df)
# nu
df = pd.DataFrame(np.random.uniform(low=0.01, high=0.3, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nu_cf= qfCreateData2D('Swaption SABR Nu', data_conv, df)
# rho
df = pd.DataFrame(np.random.uniform(low=-0.99, high=0.99, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_rho_cf = qfCreateData2D('Swaption SABR Rho', data_conv, df)

In [9]:
### create a data collection
data_list = [
    data2D_nv, data2D_beta, data2D_nu, data2D_rho,
    data2D_nv_cf, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection = qfCreateDataCollection(data_list)
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA2D,Swaption Normal Volatility,USD-SOFR-SWAPTION
1,DATA2D,Swaption SABR Beta,USD-SOFR-SWAPTION
2,DATA2D,Swaption SABR Nu,USD-SOFR-SWAPTION
3,DATA2D,Swaption SABR Rho,USD-SOFR-SWAPTION
4,DATA2D,Swaption Normal Volatility,USD-SOFR-CAPFLOOR
5,DATA2D,Swaption SABR Beta,USD-SOFR-CAPFLOOR
6,DATA2D,Swaption SABR Nu,USD-SOFR-CAPFLOOR
7,DATA2D,Swaption SABR Rho,USD-SOFR-CAPFLOOR


### Create product and display

In [10]:
caplet = qfCreateProductRFRCapletFloorlet(
    effective_date="2026-06-17",
    expiry_offset="0D",
    term_or_termination_date="3M",
    payment_date="2026-09-21",
    on_index="SOFR-1B",
    strike=0.045,
    cap_or_floor="CAP",
    notional=1_000_000,
    accrual_basis="ACT/360",
    long_or_short="LONG",
)

qfDisplayProduct(caplet)

,Name,Value
0,Product Type,PRODUCT_RFR_CAPLET_FLOORLET
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-06-17
5,Expiry Offset,0D
6,Expiry Date,2026-06-17
7,Termination Date,2026-09-17
8,Payment Date,2026-09-21
9,ON Index,SOFRON Actual/360


In [ ]:
# Please complete the product-related implementation in non_linear_products, product_factory, product_display_visitor, and the product-related API.
cap = qfCreateProductRFRCapFloor(
    effective_date="2026-06-17",
    term_or_termination_date="2Y",
    on_index="SOFR-1B",
    strike=0.045,
    cap_or_floor="CAP",
    notional=1_000_000,
    accrual_period="3M",
    accrual_basis="ACT/360",
    payment_offset="2D",
    payment_business_day_convention="F",
    payment_holiday_convention="USGS",
    long_or_short="LONG",
    business_day_convention="MF",
    holiday_convention="USGS",
)

qfDisplayProduct(cap)

,Name,Value
0,Product Type,PRODUCT_RFR_CAP_FLOOR
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-06-17
5,Termination Date,2028-06-20
6,ON Index,SOFRON Actual/360
7,Strike,0.045
8,Cap Or Floor,CAP
9,Accrual Period,3M


### Create sabr model

In [12]:
sabr_model = qfCreateSABRModel(
    sub_model=yc_model,
    data_collection=data_collection,
    build_method_collection=bm_collection
)
print("SABR components:", sabr_model.components_.keys())

SABR components: dict_keys(['SOFR-1B-SWAPTION', 'SOFR-1B-CAPFLOOR'])


In [ ]:
# You should complete SABRModelComponent.get_sabr_parameters and SABRModelComponent.get_sabr_parameter_gradient_wrt_state.
sabr_model.get_sabr_parameters(IndexRegistry().get("SOFR-1B-CAPFLOOR"), 0.5, 0.25)

{<SABRParameters.BETA: 'SWAPTION SABR BETA'>: 0.6,
 <SABRParameters.RHO: 'SWAPTION SABR RHO'>: -0.15686702432896316,
 <SABRParameters.NU: 'SWAPTION SABR NU'>: 0.2080903034459329,
 <SABRParameters.NV: 'SWAPTION NORMAL VOLATILITY'>: 0.00996725128369984}

In [13]:
vp_funding = FundingIndexParameter({"Funding Index": "SOFR-1B-FLAT"})
vpc = ValuationParametersCollection([vp_funding])
request = ValuationRequest.PV

### Price caplet
I provide the cap results here as a reference, so you can see what the expected output should look like.

In [22]:
ve_caplet = ValuationEngineRFRCapletFloorlet(
    sabr_model,
    vpc,
    caplet,
    request,
)

ve_caplet.calculate_value()

print("PV            :", ve_caplet.value_)
print("Cash          :", ve_caplet.cash_)
print("NV/BETA/NU/RHO:", ve_caplet.sabr_result_.get(SABRParameters.NV), ve_caplet.sabr_result_.get(SABRParameters.BETA), ve_caplet.sabr_result_.get(SABRParameters.NU), ve_caplet.sabr_result_.get(SABRParameters.RHO))
print("Option Value  :", ve_caplet.option_value_)

PV            : 19.651463196784384
Cash          : 0.0
NV/BETA/NU/RHO: 0.009959787334614681 0.6 0.2526865471938938 0.26123620621850296
Option Value  : 7.85413908455314e-05


In [18]:
pv_cash_report = ve_caplet.get_value_and_cash()
display(pv_cash_report.display())

,Currency,Type,Value
0,USD,PV,19.651463
1,USD,CASH,0.000000


In [19]:
cf_report = ve_caplet.create_cash_flows_report()
display(cf_report.display())

,PRODUCT_TYPE,VALUATION_ENGINE_TYPE,LEG_ID,CASHFLOW_ID,PAY_OR_RECEIVE,NOTIONAL,PAY_DATE,FORECASTED_AMOUNT,PV,DISCOUNG FACTOR,FIXING_DATE,START_DATE,END_DATE,ACCRUED,INDEX_OR_FIXED,INDEX_VALUE
0,PRODUCT_RFR_CAPLET_FLOORLET,ValuationEngineRFRCapletFloorlet,0,0,1.0,1000000,"September 21st, 2026",20.071689,19.651463,0.979064,"June 17th, 2026","June 17th, 2026","September 17th, 2026",0.255556,SOFRON Actual/360,0.03303


In [21]:
### test risk
df_risk = qfCreateValueReport(sabr_model, caplet, vpc, 'firstorderrisk').display()
df_risk[np.abs(df_risk["VALUES"].astype(float)) > 1e-10]

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
5,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,0.008402
6,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,0.765037
94,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR,3M,3M,0.00997,1.0,6131.782844
95,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR,3M,6M,0.009402,1.0,299.016643
97,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR,6M,3M,0.00995,1.0,4125.017549
98,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR,6M,6M,0.010679,1.0,201.156651
109,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR,3M,3M,0.6,1.0,4.685914
110,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR,3M,6M,0.6,1.0,0.228509
112,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR,6M,3M,0.6,1.0,3.152342
113,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR,6M,6M,0.6,1.0,0.153724


In [27]:
# Bump Reval
risk_data_type = "SWAPTION NORMAL VOLATILITY"
risk_data_convention = "USD-SOFR-CAPFLOOR"
risk_expiry = "3M"
risk_tenor = "3M"

bump_size = 1e-4
pv_base = qfCreateValueReport(sabr_model, caplet, vpc, "pv")[0][1]

In [28]:
# Step 1: build bumped up model
df_nv_cf_bumped = data2D_nv_cf.display().copy()
cur_nv = df_nv_cf_bumped.loc[risk_expiry, risk_tenor]
df_nv_cf_bumped.loc[risk_expiry, risk_tenor] = cur_nv + bump_size
data2D_nv_cf_bumped = qfCreateData2D(risk_data_type, risk_data_convention, df_nv_cf_bumped)
data_list_bumped = [data2D_nv, data2D_beta, data2D_nu, data2D_rho,
                    data2D_nv_cf_bumped, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection_bumped = qfCreateDataCollection(data_list_bumped)
sabr_model_bumped = qfCreateSABRModel(sub_model=yc_model,
                                      data_collection= data_collection_bumped,
                                      build_method_collection=bm_collection)

# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(sabr_model_bumped, caplet, vpc, 'pv')[0][1]

# Step 3 : bump-reval risk
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.10f}.')

# Step 4: analytic risk
analytic_risk = df_risk[
    (df_risk["DATA_TYPE"] == risk_data_type)
    & (df_risk["DATA_CONVENTION"] == risk_data_convention)
    & (df_risk["AXIS1"] == risk_expiry)
    & (df_risk["AXIS2"] == risk_tenor)
]['VALUES'].values[0]
analytic_scaled = analytic_risk * bump_size
print(f'Analytic risk is {analytic_scaled}')
print(f"Difference is {risk_bump_reval - analytic_scaled:.10e}.")

Bump reval risk is 0.6189505834.
Analytic risk is 0.6131782843675088
Difference is 5.7722989892e-03.


### Price CapFloor

In [ ]:
ve_cap = ValuationEngineRFRCapletFloorlet(
    sabr_model,
    vpc,
    cap,
    request,
)

ve_cap.calculate_value()

print("PV            :", ve_cap.value_)
print("Cash          :", ve_cap.cash_)
print("NV/BETA/NU/RHO:", ve_cap.sabr_result_.get(SABRParameters.NV), ve_cap.sabr_result_.get(SABRParameters.BETA), ve_cap.sabr_result_.get(SABRParameters.NU), ve_cap.sabr_result_.get(SABRParameters.RHO))
print("Option Value  :", ve_cap.option_value_)

In [ ]:
pv_cash_report_cap = ve_cap.get_value_and_cash()
display(pv_cash_report_cap.display())

In [ ]:
cf_report_cap = ve_cap.create_cash_flows_report()
display(cf_report_cap.display())

In [ ]:
### test risk
df_risk_cap = qfCreateValueReport(sabr_model, cap, vpc, 'firstorderrisk').display()
df_risk_cap[np.abs(df_risk_cap["VALUES"].astype(float)) > 1e-10]

In [ ]:
# Please write a bump reval test